In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import classification_report
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

import matplotlib.pyplot as plt
import seaborn as sns

import pickle

In [ ]:
from typing import Any, Self

class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, prefix_treshold=0.005) -> None:
        super().__init__()
        self.prefix_treshold = prefix_treshold
        self.ticket_frequency_ = {}
        self.rare_prefix_ = []

    def fit(self, X: pd.DataFrame, y: Any = None) -> Self:
        
        self.ticket_frequency_ = X['Ticket'].value_counts().to_dict()

        prefix_freq = (X['Ticket']
                        .str.replace(r'[/.]', '', regex=True)
                        .str.split()
                        .apply(lambda w: w[0] if len(w) > 1 else 'NoPrefix')
                        .value_counts(normalize=True))

        self.rare_prefix_ = prefix_freq[prefix_freq < self.prefix_treshold].index.to_list()

        return self
    
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:

        X_out = X.copy().drop(columns=['PassengerId'])

        # Cabin Feature
        X_out['HasCabin'] = X_out['Cabin'].notna().astype('int')
        X_out['Deck'] = X_out['Cabin'].str[0].fillna('U')
        X_out['CabinCount'] = ((X_out['Cabin']
                                .str.strip()
                                .str.count(' ')+1)
                                .fillna(0)
                                .astype('int'))
        X_out = X_out.drop(columns=['Cabin'])

        # Name Feature
        X_out['NameLen'] = X_out['Name'].str.len()
        X_out['Title'] = X_out['Name'].str.extract(pat=r' ([A-Za-z]+)\.', expand=False)

        flat_mapping = {
            'Mr': 'Mr',
            'Miss': 'Miss', 'Mlle': 'Miss', 'Ms': 'Miss',
            'Mrs': 'Mrs', 'Mme': 'Mrs',
            'Master': 'Master'
        }
        X_out['Title'] = X_out['Title'].map(flat_mapping).fillna('Rare').astype(str)
        X_out = X_out.drop(columns=['Name'])

        # Group Size
        local_counts = X_out['Ticket'].value_counts().to_dict()
        X_out['GroupSize'] = X_out['Ticket'].apply(
            lambda t: max(self.ticket_frequency_.get(t, 1), local_counts.get(t, 1))
        )

        # TicketPrefix
        X_out['TicketPrefix'] = (X_out['Ticket']
                                    .str.replace(r'[/.]', '', regex=True)
                                    .str.split()
                                    .apply(lambda w: w[0] if len(w) > 1 else 'NoPrefix'))
        X_out['TicketPrefix'] = (X_out['TicketPrefix']
                                    .replace(self.rare_prefix_, 'Rare')
                                    .astype(str))
        X_out = X_out.drop(columns=['Ticket'])

        cat_cols = X_out.select_dtypes(include=['object', 'category']).columns
        X_out[cat_cols] = X_out[cat_cols].astype(str)

        return X_out


In [5]:
numeric_features = ['Pclass', 'Age', 'SibSp',
                    'Parch', 'Fare', 'HasCabin',
                    'CabinCount', 'GroupSize', 'NameLen']
categorical_features = ['Sex', 'Embarked', 'Deck', 'TicketPrefix', 'Title']

In [6]:
lgb_params = {
    'learning_rate': 0.1, 
    'max_depth': 4, 
    'min_child_samples': 30, 
    'n_estimators': 200, 
    'num_leaves': 7
}

In [7]:
numeric_transformer = 'passthrough'
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

best_lgb = LGBMClassifier(
    **lgb_params,
    random_state=42,
    verbose=-1
)

pipline = Pipeline(steps=[
    ('extractor', FeatureExtractor()),
    ('preprocessor', preprocessor),
    ('classifier', best_lgb)
])

In [8]:
train_df = pd.read_csv('Data/train.csv')
y = train_df['Survived']
X = train_df.drop(columns=['Survived'])

In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)